# Whom do we call? Logistic regression on real customers

Unit 1 of *Predictive Analytics*. Work in **pairs**: one types, one writes down the predictions; swap at every station.

**How it works.** The code already runs, you do not have to write it. At every station you

1. **predict** what will happen (write it down *before* you run the cell),
2. **run** the cell and compare,
3. **change** one thing,
4. **decide** and give a reason.

**Stuck?** Every station only needs the *Setup* cell below. Just go on with the next station.

| Station | Question | Time |
|---|---|---|
| 1 | How good is doing nothing? | 8 min |
| 2 | One feature: what does the coefficient say? | 12 min |
| 3 | All features: how much better? | 10 min |
| 4 | Which threshold? | 12 min |
| 5 | Final decision | 8 min |
| Extra | For the fast ones | |

Data: Telco Customer Churn, IBM sample data (Apache License 2.0). As in the lecture, a call costs **30 €** and a saved customer is worth **100 €** (fictional).

## Setup (run once, about 10 seconds)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
FILE = Path("telco-churn.csv")
df = pd.read_csv(FILE if FILE.exists() else URL)

# 11 new customers (tenure 0) have a blank instead of a number in TotalCharges
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce").fillna(0)
df["left"] = (df["Churn"] == "Yes").astype(int)
df = df.drop(columns=["customerID", "Churn"])

# 60 % train, 20 % validation (for decisions), 20 % test (only for station 5)
train, rest = train_test_split(df, test_size=0.4, random_state=0, stratify=df["left"])
valid, _test = train_test_split(rest, test_size=0.5, random_state=0, stratify=rest["left"])

CALL_COST, SAVED = 30, 100
NUMERIC = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL = [c for c in df.columns if c not in NUMERIC + ["left"]]


def train_model(features):
    """Logistic regression on the given features, with scaling and one-hot encoding."""
    num = [f for f in features if f in NUMERIC]
    cat = [f for f in features if f in CATEGORICAL]
    prepare = make_column_transformer((StandardScaler(), num), (OneHotEncoder(drop="first"), cat))
    return make_pipeline(prepare, LogisticRegression(max_iter=2000)).fit(train[features], train["left"])


def profit_per_1000(y, called):
    """Profit per 1,000 customers: +70 for each called leaver, -30 for each other call."""
    y, called = np.asarray(y), np.asarray(called, dtype=bool)
    return 1000 * np.mean(called * (SAVED * y - CALL_COST))


def report(y, called):
    """Confusion matrix and measures for a call list."""
    tn, fp, fn, tp = confusion_matrix(y, called, labels=[0, 1]).ravel()
    print(pd.DataFrame([[tn, fp], [fn, tp]], index=["stayed", "left"], columns=["not called", "called"]), "\n")
    print(f"sensitivity {tp / (tp + fn):.2f}   specificity {tn / (tn + fp):.2f}   "
          f"precision {tp / (tp + fp) if tp + fp else float('nan'):.2f}   accuracy {(tp + tn) / len(y):.2f}")
    print(f"profit per 1,000 customers: {profit_per_1000(y, called):.0f} €")


def check(name, value, expected, tolerance=0.01):
    """Compare your value with the expected one."""
    ok = value is not None and abs(value - expected) <= tolerance
    print(f"{name}:", "correct ✅" if ok else "not yet ❌ (have another look at the output above)")


print(f"{len(train)} train, {len(valid)} validation, {len(_test)} test customers")
df.head()

## Station 1: How good is doing nothing?

**Predict:** about 27 % of the customers left. What accuracy does a "model" reach that calls **nobody**? What profit? And what happens if we call **everybody**?

**Prediction:** *write here*

In [ ]:
print("Call nobody")
report(valid["left"], np.zeros(len(valid)))
print("\nCall everybody")
report(valid["left"], np.ones(len(valid)))

**Change:** your boss says that calls are getting cheaper. From which call cost on does calling *everybody* pay off? Try values for `cost`, then enter the limit in `break_even_cost`.

In [ ]:
cost = 30  # change me
print(f"profit per 1,000 customers if we call everybody: {1000 * (SAVED * valid['left'].mean() - cost):.0f} €")

break_even_cost = None  # your result in €
check("Break-even", break_even_cost, SAVED * valid["left"].mean(), tolerance=0.6)

**Decide:** a colleague is proud of a model with 74 % accuracy. What do you answer, in one sentence?

**Answer:** *write here*

## Station 2: One feature, what does the coefficient say?

`tenure` is the number of months somebody has been a customer.

**Predict:** will the coefficient of `tenure` be positive or negative? Does one more **year** change the odds of leaving a lot or a little?

**Prediction:** *write here*

In [ ]:
feature, step = "tenure", 12  # change me later: "MonthlyCharges", 10

# Without scaling, so that the coefficient refers to one unit of the feature
m1 = LogisticRegression(max_iter=1000).fit(train[[feature]], train["left"])
b0, b1 = m1.intercept_[0], m1.coef_[0, 0]
print(f"b0 = {b0:.3f}, b1 = {b1:.4f}")
print(f"{step} units more multiply the odds of leaving by {np.exp(step * b1):.2f}")

x = np.linspace(train[feature].min(), train[feature].max(), 100)
plt.scatter(valid[feature], valid["left"] + np.random.default_rng(0).uniform(-0.04, 0.04, len(valid)), s=4, alpha=0.3, color="gray")
plt.plot(x, 1 / (1 + np.exp(-(b0 + b1 * x))), color="green", linewidth=3)
plt.xlabel(feature)
plt.ylabel("probability of leaving")
plt.show()

**Change 1:** by hand (calculator!): with which probability does a customer with `tenure` = 12 leave? Use `b0` and `b1` from the output, then check.

In [ ]:
p_12_months = None  # your result, e.g. 0.25

m_tenure = LogisticRegression(max_iter=1000).fit(train[["tenure"]], train["left"])
check("Probability", p_12_months, m_tenure.predict_proba(pd.DataFrame({"tenure": [12]}))[0, 1])

**Change 2:** set `feature, step = "MonthlyCharges", 10` in the first cell of this station and run it again.

**Decide:** complete the sentence for both features: *"... more multiply the odds of leaving by ..."*. And why does the curve not look like an S?

**Answer:** *write here*

## Station 3: All features, how much better?

**Predict:** the model with `tenure` alone has an AUC of about 0.75 on the validation data. What AUC do you expect with **all** features?

**Prediction:** *write here*

In [ ]:
features = NUMERIC + CATEGORICAL  # change me: e.g. ["tenure", "Contract"]

m3 = train_model(features)
p = m3.predict_proba(valid[features])[:, 1]

fpr, tpr, _ = roc_curve(valid["left"], p)
plt.plot(fpr, tpr, color="green", linewidth=3, label=f"AUC = {roc_auc_score(valid['left'], p):.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("1 - specificity")
plt.ylabel("sensitivity")
plt.legend()
plt.show()

Which features drive the prediction? Values above 1 increase the odds of leaving, values below 1 decrease them. Numeric features are scaled, so the factor refers to one standard deviation; categories are compared with the first category, which is left out.

In [ ]:
m_all = train_model(NUMERIC + CATEGORICAL)
names = [n.split("__")[1] for n in m_all[0].get_feature_names_out()]
odds_factor = pd.Series(np.exp(m_all[-1].coef_[0]), index=names).sort_values()
pd.concat([odds_factor.head(4), odds_factor.tail(4)]).round(2)

**Change:** find **three** features with which you get as close as possible to the AUC of all features.

**Decide:** which three would you take, and what do you gain from a smaller model?

**Answer:** *write here*

## Station 4: Which threshold?

**Predict:** at which threshold will the profit be highest on the validation data?

**Prediction:** *write here*

In [ ]:
m4 = train_model(NUMERIC + CATEGORICAL)
p4 = m4.predict_proba(valid[NUMERIC + CATEGORICAL])[:, 1]

thresholds = np.arange(0, 1.01, 0.01)
profits = [profit_per_1000(valid["left"], p4 >= t) for t in thresholds]
plt.plot(thresholds, profits, color="green", linewidth=3)
plt.axhline(0, color="gray", linewidth=1)
plt.xlabel("threshold")
plt.ylabel("profit per 1,000 customers in €")
plt.show()

In [ ]:
threshold = 0.5  # change me

report(valid["left"], p4 >= threshold)

**Change:** compare 0.5 with 0.3. Which measures get better, which get worse?

**Decide:** the accuracy is higher at 0.5, the profit at 0.3. Which one do you recommend, and to whom would you have to explain it?

**Answer:** *write here*

## Station 5: Final decision

Enter your features and your threshold. `final_test` uses the test data that nobody has seen so far. You have **one shot**: whoever tries several thresholds on the test data has used it for training.

In [ ]:
my_features = NUMERIC + CATEGORICAL
my_threshold = 0.5  # your decision

m5 = train_model(my_features)


def final_test():
    """Evaluate your decision on the test data. One shot!"""
    report(_test["left"], m5.predict_proba(_test[my_features])[:, 1] >= my_threshold)


# final_test()   # remove the first # when you are sure

**Hand in (two sentences):** which threshold did you choose and why? What profit do you promise your boss, and why not more?

**Answer:** *write here*

## Extra: for the fast ones

1. **Limited capacity:** the call centre can only make 100 calls on the validation data. Whom do you call? How many of them really leave?
2. **Not everybody can be convinced:** assume only half of the called leavers stay. What is the new break-even threshold? Adapt `profit_per_1000`.
3. **Exam question:** explain in two sentences why the AUC does not change when you change the threshold.

In [ ]:
# your space